# Example 10.1.3 / 2: Side Variables in Node Balances

Companion notebook for **10.1.3 Multicommodity Flows and Side Constraints** in the AMPL lab Documentation, section "Side Variables in Node Balances". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Side Variables in Node Balances

An ordinary `var` can appear in a `node` condition. The book calls such variables side variables. Here the plant has no fixed supply: it converts feedstocks into products, and one unit of feedstock `f` yields `yield[p, f]` units of each product `p`. Because `Feed` appears in the balance of every product, it ties the product networks together just as a shared capacity does.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set CITIES;  # network nodes
set LINKS within CITIES cross CITIES;  # directed shipping links
set PRODS;  # commodities sharing the links
set FEEDS;  # feedstocks converted into products at a city

param demand {CITIES, PRODS} >= 0 default 0;   # units required
param cost {LINKS, PRODS} >= 0;                # cost per unit shipped
param yield {PRODS, FEEDS} >= 0;               # product units per feedstock unit
param limit {FEEDS, CITIES} >= 0 default 0;    # feedstock available

minimize TotalCost;  # terms are added by the arc declarations

var Feed {f in FEEDS, k in CITIES} >= 0, <= limit[f, k];  # side variable: feedstock used at k

node Balance {k in CITIES, p in PRODS}:  # per-product balance at each city
    net_out = sum {f in FEEDS} yield[p, f] * Feed[f, k] - demand[k, p];

# flow of product p on link (i, j), uncapacitated
arc Ship {(i, j) in LINKS, p in PRODS} >= 0,
    from Balance[i, p], to Balance[j, p],
    obj TotalCost cost[i, j, p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["CITIES"] = ["plant", "east"]
ampl.set["LINKS"] = [("plant", "east")]
ampl.set["PRODS"] = ["bands", "coils"]
ampl.set["FEEDS"] = ["ore", "scrap"]
ampl.param["demand"] = {("east", "bands"): 4, ("east", "coils"): 3}
ampl.param["cost"] = {("plant", "east", "bands"): 1, ("plant", "east", "coils"): 1}
ampl.param["yield"] = {("bands", "ore"): 2, ("coils", "ore"): 1, ("bands", "scrap"): 1, ("coils", "scrap"): 1}
ampl.param["limit"] = {("ore", "plant"): 3, ("scrap", "plant"): 3}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Feed"].to_dict())
print(ampl.get_data("Ship").to_dict())

HiGHS 1.14.0: optimal solution; objective 7
0 simplex iterations
0 barrier iterations
solved
{('ore', 'east'): 0, ('ore', 'plant'): 1, ('scrap', 'east'): 0, ('scrap', 'plant'): 2}
{('plant', 'east', 'bands'): 4, ('plant', 'east', 'coils'): 3}


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
set CITIES := plant east;
set LINKS := (plant,east);
set PRODS := bands coils;
set FEEDS := ore scrap;
param demand := [east,bands] 4 [east,coils] 3;
param cost := [plant,east,bands] 1 [plant,east,coils] 1;
param yield: ore scrap :=
    bands  2 1
    coils  1 1;
param limit := [ore,plant] 3 [scrap,plant] 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Feed, Ship;

HiGHS 1.14.0: optimal solution; objective 7
0 simplex iterations
0 barrier iterations
Feed :=
ore   east    0
ore   plant   1
scrap east    0
scrap plant   2
;

Ship :=
plant east bands   4
plant east coils   3
;

